In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Nov  5 14:07:15 2025

@author: 1
"""

# -*- coding: utf-8 -*-
"""
"""

import os
import numpy as np
import rasterio
from datetime import datetime
from tqdm import tqdm
import glob

# ===================== 参数设置 =====================
data_dir = r"......\DTR\daily"
output_file = r"......\DTR\TNin90.tif"

start_year, end_year = 1981, 2010
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# ===================== 构建每日文件索引 =====================
# 获取所有tif文件路径
all_files = sorted(glob.glob(os.path.join(data_dir, "DTR*.tif")))

# 筛选出基准期文件
def valid_date(fn):
    try:
        date_str = os.path.basename(fn)[3:11]  # 提取 YYYYMMDD
        date = datetime.strptime(date_str, "%Y%m%d")
        return start_year <= date.year <= end_year and not (date.month == 2 and date.day == 29)
    except:
        return False

all_files = [f for f in all_files if valid_date(f)]
print(f"找到 {len(all_files)} 个有效日文件")

# ===================== 读取第一个文件获取元数据 =====================
with rasterio.open(all_files[0]) as src:
    meta = src.meta.copy()
    height, width = src.height, src.width
    transform, crs = src.transform, src.crs
    nodata = src.nodata

# 初始化结果数组：365天 × 高 × 宽
tnin90 = np.full((365, height, width), np.nan, dtype=np.float32)

# 初始化按日序存储的容器
all_data = {d: [] for d in range(365)}

# ===================== 逐文件读取并按日序整理 =====================
for file in tqdm(all_files, desc="Reading daily DTR"):
    date_str = os.path.basename(file)[3:11]
    date = datetime.strptime(date_str, "%Y%m%d")
    day_of_year = (date - datetime(date.year, 1, 1)).days  # 0-based (0~364)
    if day_of_year >= 365:
        continue  # 跳过2月29日

    with rasterio.open(file) as src:
        data = src.read(1).astype(np.float32)
        if nodata is not None:
            data[data == nodata] = np.nan
        all_data[day_of_year].append(data)

# ===================== 计算第90百分位（5日滑动窗口） =====================
for day in tqdm(range(365), desc="Computing TNin90"):
    window_data = []
    for offset in range(-7, 8):  # ±2天窗口
        day_idx = (day + offset) % 365  # 环绕
        window_data.extend(all_data[day_idx])
    if len(window_data) == 0:
        continue
    window_stack = np.stack(window_data, axis=0)
    tnin90[day] = np.nanpercentile(window_stack, 90, axis=0)

# ===================== 保存输出 =====================
meta.update({
    "count": 365,
    "dtype": "float32",
    "compress": "lzw",
    "nodata": -9999
})

with rasterio.open(output_file, "w", **meta) as dst:
    for day in range(365):
        out_data = np.where(np.isnan(tnin90[day]), -9999, tnin90[day])
        dst.write(out_data, day + 1)
        dst.set_band_description(day + 1, f"Day-{day + 1}")

print("✅ TNin90 calculation completed.")
print("Output saved to:", output_file)
